In [1]:
import pickle
import numpy as np


### INCARCAREA DATELOR
class NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core")
        return super().find_class(module, name)
if np.__version__ >= '2.0.0':
    with open('train_data.pkl', 'rb') as f:
        data = pickle.load(f)
else:
    with open('train_data.pkl', 'rb') as f:
        data = NumpyCompatUnpickler(f).load()

train_image_embeddings = data['train_image_embeddings']
gender_labels = data['gender_labels']
test_image_embeddings = data['test_image_embeddings']

print('-----DATA INFO-----')
print('Shape of train_image_embeddings:', train_image_embeddings.shape)
print('Shape of gender_labels:', gender_labels.shape)
print('Shape of test_image_embeddings:', test_image_embeddings.shape)
print('-----DATA INFO-----')


### GENERAREA SUBMISIEI - NU MODIFICATI CODUL DE AICI. MODIFICAREA CODULUI VA REZULTA INTR-O SUBMISIE INVALIDA.
# def generate_submission(debiased_train_image_embeddings, debiased_test_image_embeddings, compression=data['compression']):
#     train_compression = debiased_train_image_embeddings @ compression.T
#     test_compression = debiased_test_image_embeddings @ compression.T
#     with open('submission.pkl', 'wb') as f:
#         pickle.dump({'train_compression': train_compression, 'test_compression': test_compression}, f)
# generate_submission(debiased_train_image_embeddings, debiased_test_image_embeddings)

-----DATA INFO-----
Shape of train_image_embeddings: (15000, 512)
Shape of gender_labels: (15000,)
Shape of test_image_embeddings: (15000, 512)
-----DATA INFO-----


In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

train_embs, val_embs, train_labels, val_labels = train_test_split(train_image_embeddings, gender_labels, test_size=0.1)
train_embs, val_embs = torch.tensor(train_embs), torch.tensor(val_embs)
train_labels, val_labels = torch.tensor(train_labels, dtype=torch.float32).reshape(-1, 1), torch.tensor(val_labels, dtype=torch.float32).reshape(-1, 1), 
ds_train = TensorDataset(train_embs, train_labels)
ds_val = TensorDataset(val_embs, val_labels)
train_loader = DataLoader(ds_train, batch_size=16, shuffle=True)
val_loader = DataLoader(ds_val, batch_size=16)

In [14]:
from torch import nn
import torch.nn.functional as F
class FC(nn.Module):
    def __init__(self):
        super().__init__()
        # self.fc1 = nn.Linear(512, 256)
        # self.fc2 = nn.Linear(256, 64)
        # self.fc3 = nn.Linear(64, 16)
        # self.fc4 = nn.Linear(16, 1)
        self.fc1 = nn.Linear(512, 1)
    
    def forward(self, x):
        # return F.sigmoid(self.fc4(F.relu(self.fc3(F.relu(self.fc2(F.relu(self.fc1(x))))))))
        return F.sigmoid(self.fc1(x))
    
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = FC().to(device)

In [16]:
from tqdm import tqdm
epochs = 30
criterion = nn.BCELoss()
optim = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(epochs):
    train_loss = 0.0
    train_acc = 0.0
    for (emb, label) in train_loader:
        emb, label = emb.to(device), label.to(device)
        pred = model(emb)
        loss = criterion(pred, label)

        loss.backward()
        optim.step()
        optim.zero_grad()

        train_loss += loss.item() * len(label)
        train_acc += ((pred>0.5) == label).sum()

    train_loss/=len(ds_train)
    train_acc/=len(ds_train)

    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for (emb, label) in val_loader:
            emb, label = emb.to(device), label.to(device)
            pred = model(emb)
            loss = criterion(pred, label)

            val_loss += loss.item() * len(label)
            val_acc += ((pred>0.5) == label).sum()

    val_loss/=len(ds_val)
    val_acc/=len(ds_val)

    print(f'Epoch: {epoch}/{epochs} | Train loss: {train_loss} | Val loss: {val_loss} | Train Acc: {train_acc} | Val Acc: {val_acc}')

Epoch: 0/30 | Train loss: 0.384498305356061 | Val loss: 0.36439789430300396 | Train Acc: 0.980222225189209 | Val Acc: 0.9886666536331177
Epoch: 1/30 | Train loss: 0.3446814452807109 | Val loss: 0.32734631554285687 | Train Acc: 0.9892592430114746 | Val Acc: 0.9913333058357239


KeyboardInterrupt: 